# Multi-rate evaluation speed benchmark

Expanded $100B run, September 11, 2026. Compare commit `896fad1` with the optimized evaluator using the completed epoch-one weights. The first panel has the same 384 observations across the broad universe; the second contains 128 consecutive anchors each for AAPL, MSFT and NVDA. Reference batches contain 32 rows in date/symbol order; optimized batches contain 128 in symbol/date order. Both preserve per-symbol chronology and all inputs, targets and scores.

Measured scoring-loop times include materialization, model forward and the full deduplicated NTP persistence audit, with first-batch overhead. They exclude startup/metadata construction and the portfolio simulation. These are representative samples, not whole-run timing guarantees. The saved predictions compare every time position of all 26 supervised heads, not just final trading scores. All daily HITS rankings in both panels match, NTP counts and persistence baselines match exactly, and prediction differences are at most 1.2e-6. The vectorized successor lookup also has exact tests for gaps, missing families, repeated dates and singleton histories.

The full run scores 96,181 windows, versus 14,849 spaced training windows per epoch. Full evaluation remains more work than an equal-sized inference pass. The monitor now records inference and portfolio-backtest durations separately.

Optimizations: skip unused label/document work; group unchanged annual/quarterly contexts; preserve disk-backed Polars input locality; bulk-transfer audit columns and deduplicate recurring pairs in bounded memory; vectorize successor indices; free unused training CUDA reservations at epoch gates. Model weights, feature coverage, training batch size, FP32 precision, scoring dates, original anchored HITS rules, adjusted-price treatment and costs are unchanged.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
os.chdir(repo)
root = repo / "artifacts/multirate_recovery/100B/evaluation_speed_benchmark"
root.mkdir(parents=True, exist_ok=True)
reference = root / "reference"
reference.mkdir(exist_ok=True)
for source in ("scripts/train_multirate_mtl.py", "quant_orchestrator/research_tools/ntp_evaluation.py", "quant_orchestrator/research_tools/multirate_objectives.py"):
    (reference / Path(source).name).write_bytes(subprocess.check_output(["git", "show", f"896fad1:{source}"]))


In [ ]:
runner_source = "import sys,json,runpy,time,os,resource,builtins\nfrom pathlib import Path\nimport torch\nmode=sys.argv[1];root=Path('artifacts/multirate_recovery/100B').resolve();out=root/'evaluation_speed_benchmark'/mode;out.mkdir(parents=True,exist_ok=True)\nif mode.startswith('baseline'):\n import importlib.util\n spec=importlib.util.spec_from_file_location('quant_orchestrator.research_tools.multirate_objectives',root/'evaluation_speed_benchmark/reference/multirate_objectives.py');reference_objectives=importlib.util.module_from_spec(spec);sys.modules[spec.name]=reference_objectives;spec.loader.exec_module(reference_objectives)\n import quant_orchestrator.research_tools.ntp_evaluation as audit_module\n spec=importlib.util.spec_from_file_location('quant_orchestrator.research_tools._reference_audit',str(root/'evaluation_speed_benchmark/reference/ntp_evaluation.py'));reference=importlib.util.module_from_spec(spec);spec.loader.exec_module(reference)\n audit_module.NTPPersistenceAudit=reference.NTPPersistenceAudit\nfrom quant_orchestrator.research_tools.ntp_evaluation import NTPPersistenceAudit\nfrom quant_orchestrator.research_tools.multirate_batch import BatchTensors\nfrom quant_orchestrator.platforms.ml_frameworks.torch.models.transformers.multirate.model import MultiRateTransformer\nmetrics={'stage_seconds':0.,'forward_seconds':0.,'audit_seconds':0.,'audit_calls':0};records=[];scores=[]\nold_stage=BatchTensors.__call__;old_forward=MultiRateTransformer.forward;old_audit=NTPPersistenceAudit.update\nlast=None\ncurrent_keys=[]\nobservations=0\ndef stage(self,*a,**kw):\n global last,current_keys\n current_keys=[(item['symbol'],item['date']) for item in self.batch]\n if last is None:last=time.perf_counter()\n t=time.perf_counter();r=old_stage(self,*a,**kw);metrics['stage_seconds']+=time.perf_counter()-t;return r\ndef forward(self,*a,**kw):\n torch.cuda.synchronize();t=time.perf_counter();r=old_forward(self,*a,**kw);torch.cuda.synchronize();metrics['forward_seconds']+=time.perf_counter()-t\n scores.append({'keys':current_keys,'scores':{k:v.detach().cpu() for k,v in r['token_outputs'].items()}});return r\ndef audit(self,*a,**kw):\n global last,observations\n torch.cuda.synchronize();t=time.perf_counter();r=old_audit(self,*a,**kw);torch.cuda.synchronize();metrics['audit_seconds']+=time.perf_counter()-t;metrics['audit_calls']+=1\n if metrics['audit_calls']==4:\n  now=time.perf_counter();records.append(dict(metrics,seconds=now-last,observations=len(current_keys)));print(json.dumps(records[-1]),flush=True)\n  (out/'measurements.json').write_text(json.dumps({'batches':records,'peak_rss_mib':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024},indent=2));metrics.update(stage_seconds=0.,forward_seconds=0.,audit_seconds=0.,audit_calls=0);last=now\n  observations+=len(current_keys)\n  if observations>=384:\n   torch.save(scores,out/'scores.pt');(out/'ntp_report.json').write_text(json.dumps(self.report()));self.close();raise SystemExit(0)\n return r\nBatchTensors.__call__=stage;MultiRateTransformer.forward=forward;NTPPersistenceAudit.update=audit\ncmd=json.loads((root/'training_command_expanded_v9.json').read_text());cmd[cmd.index('--output-dir')+1]=str(out)\ni=cmd.index('--epoch-evaluation-dir');del cmd[i:i+2]\ncmd+=['--checkpoint',str(root/'train_expanded_v9/epoch_validation_2024_2026/epoch_0001/model.pt'),'--max-samples','0','--inference-only']\nif not mode.startswith('baseline'):cmd[cmd.index('--batch-size')+1]='64' if '64' in mode else '128'\ndef benchmark_sorted(items,*args,**kwargs):\n if isinstance(items,list) and len(items)>50000 and isinstance(items[0],dict) and 'symbol' in items[0] and 'date' in items[0]:\n  if mode.endswith('_local'):\n   items=[item for symbol in ('AAPL','MSFT','NVDA') for item in builtins.sorted([r for r in items if r['symbol']==symbol],key=lambda r:r['date'])[100:228]]\n  else:\n   items=builtins.sorted(items,key=lambda item:(item['date'],item['symbol']))[60000:60384]\n return builtins.sorted(items,*args,**kwargs)\nsys.argv=cmd[1:]\nrunpy.run_path(str(root/'evaluation_speed_benchmark/reference/train_multirate_mtl.py') if mode.startswith('baseline') else cmd[1],run_name='__main__',init_globals={'sorted':benchmark_sorted})\n"
(root / "benchmark.py").write_text(runner_source)
# Each subprocess uses the same immutable epoch-one checkpoint and FP32 CUDA.
# Run sequentially; concurrent GPU benchmarks would invalidate the timings.
RUN_BENCHMARKS = False
if RUN_BENCHMARKS:
    for mode in ("baseline_384", "optimized_final_384", "baseline_local", "optimized_scan_local"):
        with (root / f"{mode}.log").open("w") as log:
            subprocess.run([sys.executable, str(root / "benchmark.py"), mode], check=True,
                           stdout=log, stderr=subprocess.STDOUT,
                           env={**os.environ, "PYTHONPATH": str(repo) + os.pathsep + str(repo.parent / "quant-warehouse")})


In [1]:
import json, math
from pathlib import Path
import torch
root=Path('artifacts/multirate_recovery/100B/evaluation_speed_benchmark')
def compare(before, after):
    def read(name):
        return {(str(key[0]),str(key[1])): {task: tensor[i] for task,tensor in batch['scores'].items()}
                for batch in torch.load(root/name/'scores.pt',weights_only=False)
                for i,key in enumerate(batch['keys'])}
    a,b=read(before),read(after)
    assert a.keys()==b.keys()
    maximum=0.
    for key in a:
        for task in a[key]:
            torch.testing.assert_close(a[key][task],b[key][task],atol=2e-5,rtol=2e-5)
            maximum=max(maximum,(a[key][task]-b[key][task]).abs().max().item())
    reports=[json.loads((root/name/'ntp_report.json').read_text()) for name in (before,after)]
    for x,y in zip(reports[0]['metrics'],reports[1]['metrics'],strict=True):
        for key,value in x.items():
            if key in ('model_mse','skill') and value is not None:
                assert math.isclose(value,y[key],rel_tol=1e-5,abs_tol=1e-5)
            else:
                assert value==y[key],(key,value,y[key])
    for task in next(iter(a.values())):
        if task.startswith('hits_'):
            for date in {key[1] for key in a}:
                keys=[key for key in a if key[1]==date]
                assert sorted(keys,key=lambda k:(float(a[k][task][-1]),k[0])) == sorted(keys,key=lambda k:(float(b[k][task][-1]),k[0]))
    seconds={name:sum(batch['seconds'] for batch in json.loads((root/name/'measurements.json').read_text())['batches']) for name in (before,after)}
    return dict(baseline=before,optimized=after,observations=len(a),seconds=seconds,speedup=seconds[before]/seconds[after],max_abs_prediction_difference=maximum,all_hits_daily_ranks_identical=True,ntp_counts_and_baselines_exact=True,model_metrics_rtol=1e-5,model_metrics_atol=1e-5)
results=[compare('baseline_384','optimized_final_384'), compare('baseline_local','optimized_scan_local')]
(root/'comparison.json').write_text(json.dumps(results,indent=2))
print(json.dumps(results,indent=2))


[
  {
    "baseline": "baseline_384",
    "optimized": "optimized_final_384",
    "observations": 384,
    "seconds": {
      "baseline_384": 43.46245934300532,
      "optimized_final_384": 15.61751201399602
    },
    "speedup": 2.7829310650797234,
    "max_abs_prediction_difference": 1.1920928955078125e-06,
    "all_hits_daily_ranks_identical": true,
    "ntp_counts_and_baselines_exact": true,
    "model_metrics_rtol": 1e-05,
    "model_metrics_atol": 1e-05
  },
  {
    "baseline": "baseline_local",
    "optimized": "optimized_scan_local",
    "observations": 384,
    "seconds": {
      "baseline_local": 26.906612043007044,
      "optimized_scan_local": 11.807169321997208
    },
    "speedup": 2.2788368074707797,
    "max_abs_prediction_difference": 9.5367431640625e-07,
    "all_hits_daily_ranks_identical": true,
    "ntp_counts_and_baselines_exact": true,
    "model_metrics_rtol": 1e-05,
    "model_metrics_atol": 1e-05
  }
]
